# Tutorial 06 — Hydrometeor mixtures at C-band

Real radar volumes almost always contain more than one species —
rain with melting ice, pristine crystals with aggregates, graupel
in the mixed phase. The combined polarimetric signature is the
incoherent sum of the per-species S and Z matrices; the
*non*-linear observables (Z_dr, ρ_hv, δ_hv) cannot be averaged
from per-species values and must be computed from the mixture's
summed amplitude and phase matrices.

`HydroMix` takes a list of `MixtureComponent(Scatterer, PSD)` pairs
and exposes a Scatterer-shaped API so the usual `radar.*` helpers
work on it directly.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from rustmatrix import (HydroMix, MixtureComponent,
                         Scatterer, radar, psd as rs_psd)
from rustmatrix.tmatrix_aux import (dsr_thurai_2007, geom_horiz_back,
                                      geom_horiz_forw, K_w_sqr, wl_C)
from rustmatrix.refractive import m_w_10C, mi


## Build one scatterer per species

Each species gets its own `PSDIntegrator` that tabulates S(D)
and Z(D) for whatever refractive index, axis ratio, and orientation
PDF the species requires. All integrators must register the same
geometries the mixture will query and must share the wavelength.


In [ ]:
def build_rain():
    s = Scatterer(wavelength=wl_C, m=m_w_10C[wl_C],
                  Kw_sqr=K_w_sqr[wl_C], ddelt=1e-4, ndgs=2)
    integ = rs_psd.PSDIntegrator()
    integ.D_max = 6.0; integ.num_points = 64
    integ.axis_ratio_func = lambda D: 1.0 / dsr_thurai_2007(D)
    integ.geometries = (geom_horiz_back, geom_horiz_forw)
    s.psd_integrator = integ
    s.psd_integrator.init_scatter_table(s)
    return s

def build_ice():
    s = Scatterer(wavelength=wl_C, m=mi(wl_C, 0.2),
                  Kw_sqr=K_w_sqr[wl_C], axis_ratio=0.6,
                  ddelt=1e-4, ndgs=2)
    integ = rs_psd.PSDIntegrator()
    integ.D_max = 8.0; integ.num_points = 64
    integ.geometries = (geom_horiz_back, geom_horiz_forw)
    s.psd_integrator = integ
    s.psd_integrator.init_scatter_table(s)
    return s

rain = build_rain(); ice = build_ice()
rain_psd = rs_psd.GammaPSD(D0=1.5, Nw=8e3, mu=4, D_max=6.0)
ice_psd  = rs_psd.ExponentialPSD(N0=5e3, Lambda=2.0, D_max=8.0)
rain.psd = rain_psd; ice.psd = ice_psd


## Assemble the mixture and read the observables


In [ ]:
mix = HydroMix([
    MixtureComponent(rain, rain_psd, 'rain'),
    MixtureComponent(ice,  ice_psd,  'ice'),
])

def obs(x):
    x.set_geometry(geom_horiz_back)
    out = dict(Zh=10*np.log10(radar.refl(x)),
               Zdr=10*np.log10(radar.Zdr(x)),
               rho=radar.rho_hv(x))
    x.set_geometry(geom_horiz_forw)
    out['Kdp'] = radar.Kdp(x); out['Ai'] = radar.Ai(x)
    return out

for name, obj in [('rain only', rain),
                  ('ice only',  ice),
                  ('mixture',   mix)]:
    o = obs(obj)
    print(f'{name:<12} Z_h={o["Zh"]:6.2f}  Z_dr={o["Zdr"]:+.3f}  '
          f'ρ_hv={o["rho"]:.5f}  K_dp={o["Kdp"]:+.3f}  A_i={o["Ai"]:.4f}')


## Sweep the ice fraction

Scale the ice PSD's N0 from zero up to the rain contribution and
watch Z_dr, ρ_hv, and K_dp interpolate between the rain-only and
ice-heavy limits. Because both S and Z are linear in N(D), this
is exactly the behaviour predicted by incoherent mixing.


In [ ]:
N0_ice_sweep = np.linspace(0.0, 2e4, 9)
Zh_sw, Zdr_sw, rho_sw, Kdp_sw = [], [], [], []
for N0 in N0_ice_sweep:
    ice_psd_sw = rs_psd.ExponentialPSD(N0=float(N0), Lambda=2.0, D_max=8.0)
    m_sw = HydroMix([
        MixtureComponent(rain, rain_psd, 'rain'),
        MixtureComponent(ice,  ice_psd_sw, 'ice'),
    ])
    o = obs(m_sw)
    Zh_sw.append(o['Zh']); Zdr_sw.append(o['Zdr'])
    rho_sw.append(o['rho']); Kdp_sw.append(o['Kdp'])

fig, axes = plt.subplots(2, 2, figsize=(9, 6), sharex=True)
axes[0, 0].plot(N0_ice_sweep, Zh_sw,  'C0-o'); axes[0, 0].set_ylabel('Z_h [dBZ]')
axes[0, 1].plot(N0_ice_sweep, Zdr_sw, 'C1-o'); axes[0, 1].set_ylabel('Z_dr [dB]')
axes[1, 0].plot(N0_ice_sweep, rho_sw, 'C2-o'); axes[1, 0].set_ylabel('ρ_hv')
axes[1, 1].plot(N0_ice_sweep, Kdp_sw, 'C3-o'); axes[1, 1].set_ylabel('K_dp [°/km]')
for ax in axes.flat:
    ax.set_xlabel('ice N₀ [m⁻³ mm⁻¹]')
    ax.grid(True, alpha=0.3)
fig.suptitle('C-band rain + oriented ice — sweeping the ice concentration')
fig.tight_layout();
